# IE4483 Sentiment Analysis: TF-IDF + Logistic Regression baseline

This notebook follows the preprocessing in `tfidf_features_example`: a **default `TfidfVectorizer()`** fitted on the training data only, then used unchanged to transform the test data. It then trains and evaluates a logistic regression classifier.

**Note:** `test.json` has **no `sentiments` labels**, so accuracy can't be computed on it. Accuracy is measured on a labelled 20% validation split of `train.json` (plus 5-fold cross-validation). The final model is retrained on all of `train.json` and predicts `test.json`, and those predictions are saved for the submission.

Sections: 1. Read data · 2. Explore · 3. Preprocessing (TF-IDF) · 4. Train/validation split · 5. Training · 6. Validation results · 7. Cross-validation · 8. Error analysis · 9. Testing on `test.json`

## Import the necessary libraries

In [ ]:
import html, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (accuracy_score, f1_score, classification_report,
                             confusion_matrix, roc_auc_score, roc_curve)

SEED = 42
NEG, POS = '#eb6834', '#2a78d6'          # negative = orange, positive = blue (used consistently)
INK, MUTED, GRID = '#0b0b0b', '#52514e', '#e6e5e0'
plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False,
                     'axes.edgecolor': MUTED, 'axes.labelcolor': INK, 'xtick.color': MUTED, 'ytick.color': MUTED,
                     'axes.grid': True, 'grid.color': GRID, 'grid.linewidth': 0.8, 'axes.axisbelow': True,
                     'font.size': 10})
pd.set_option('display.max_colwidth', 120)

## 1. Read the data to pandas DataFrames

In [ ]:
import sys
sys.path.insert(0, 'Packages')                         # the project's helpers (Code/Packages)
from data_loader import load_train, load_test       # data/ or IE4483_DATA (see README)
train_df = load_train()
test_df = load_test()
print('train:', train_df.shape, list(train_df.columns))
print('test: ', test_df.shape, list(test_df.columns), '<- no sentiments column, so no labels')
train_df.head()

## 2. Explore the data

In [ ]:
counts = train_df['sentiments'].value_counts().sort_index()
lengths = train_df['reviews'].str.split().str.len()

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
ax = axes[0]
bars = ax.bar(['Negative (0)', 'Positive (1)'], counts.values, color=[NEG, POS], width=0.5)
for b, v in zip(bars, counts.values):
    ax.text(b.get_x() + b.get_width() / 2, v + 80, f'{v:,} ({v / counts.sum():.1%})', ha='center', color=INK)
ax.set_title('Class balance in train.json', loc='left', color=INK)
ax.set_ylabel('Reviews'); ax.grid(axis='x', visible=False)

ax = axes[1]
bins = np.arange(0, 305, 10)
for lab, col, name in [(1, POS, 'Positive'), (0, NEG, 'Negative')]:
    ax.hist(lengths[train_df.sentiments == lab].clip(upper=300), bins=bins, color=col, alpha=0.75,
            density=True, label=name, edgecolor='white', linewidth=0.5)
ax.set_title('Review length (words, capped at 300)', loc='left', color=INK)
ax.set_xlabel('Words per review'); ax.set_ylabel('Share of reviews'); ax.legend(frameon=False)
plt.tight_layout(); plt.show()

print(f'Majority-class baseline (always predict positive): {counts[1] / counts.sum():.2%} accuracy')
print(lengths.groupby(train_df.sentiments).describe()[['mean', '50%', 'max']].rename(index={0: 'negative', 1: 'positive'}))

## 3. Preprocessing: convert text to TF-IDF vectors

The example uses `TfidfVectorizer()` with its **default settings**:
- **lowercase** all text
- **tokenise** with the pattern `(?u)\b\w\w+\b`: words of 2 or more letters/digits. Punctuation is dropped, single characters like `"I"` or `"a"` are dropped, and `"don't"` becomes `"don"` (the `t` is lost)
- **no stop-word removal**, **unigrams only** (single words), every word kept (`min_df=1`)
- weight each word with **TF-IDF**: term count × smoothed inverse document frequency (rare words weigh more), then scale each review's vector to unit length (L2 norm)

The cell below shows what the tokeniser does to one review.

In [ ]:
vectorizer = TfidfVectorizer()          # identical to the example
analyzer = vectorizer.build_analyzer()

sample = "I didn't like it AT ALL!! The zipper broke after 2 days... A real waste of $20."
print('raw:   ', sample)
print('tokens:', analyzer(sample))

### Fit on the training data only
As in the example, the vectorizer is fitted only on training data, because the test data are unknown during training. Fitting means learning the vocabulary and IDF weights.

In [ ]:
vectorizer = vectorizer.fit(train_df['reviews'])
X_train_full = vectorizer.transform(train_df['reviews'])
X_test = vectorizer.transform(test_df['reviews'])          # same fitted vectorizer
print('X_train:', X_train_full.shape)
print('X_test: ', X_test.shape)
print(f'Sparsity: on average a review uses {X_train_full.getnnz(axis=1).mean():.1f} of '
      f'{X_train_full.shape[1]:,} vocabulary words ({X_train_full.nnz / np.prod(X_train_full.shape):.3%} non-zero)')

### What one review looks like as a TF-IDF vector

In [ ]:
vocab = vectorizer.get_feature_names_out()
i = 0
row = X_train_full[i]
vec_df = (pd.DataFrame({'word': vocab[row.indices], 'tf-idf': row.data,
                        'idf': vectorizer.idf_[row.indices]})
            .sort_values('tf-idf', ascending=False).reset_index(drop=True))
print('Review:', train_df.reviews[i])
print('Label: ', train_df.sentiments[i])
vec_df.round(3)

In [ ]:
# IDF: common words (low IDF) get small weights, rare words (high IDF) get large ones
idf = pd.Series(vectorizer.idf_, index=vocab).sort_values()
fig, ax = plt.subplots(figsize=(11, 3.4))
ax.hist(vectorizer.idf_, bins=40, color=POS, edgecolor='white', linewidth=0.5)
ax.set_title('IDF weights across the vocabulary', loc='left', color=INK)
ax.set_xlabel('IDF value'); ax.set_ylabel('Number of words'); ax.grid(axis='x', visible=False)
plt.tight_layout(); plt.show()
print('Lowest IDF (most common):', ', '.join(idf.index[:12]))
print('Highest IDF (rarest):    ', ', '.join(idf.index[-12:]))

## 4. Train / validation split

Since `test.json` is unlabelled, 20% of `train.json` is held out as a **validation set**. The split is
- **stratified**: both parts keep the same 85/15 positive/negative ratio
- **grouped**: duplicate reviews (about 220 in train) always land on the same side, so the model is never scored on a review it was trained on

`StratifiedGroupKFold` with 5 folds is used, and fold 0 is the validation set.

In [ ]:
def group_key(text):
    # duplicate detection: ignore case, whitespace and punctuation
    t = html.unescape(str(text)).lower().replace('\u2019', "'")
    return ' '.join(re.findall(r"[a-z0-9']+", t))

X_text = train_df['reviews'].values
y = train_df['sentiments'].values
groups = train_df['reviews'].map(group_key).values

folds = list(StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED).split(X_text, y, groups))
tr_idx, va_idx = folds[0]
assert not set(groups[tr_idx]) & set(groups[va_idx]), 'duplicate leaked across split'

print(f'train part: {len(tr_idx)} reviews, positive rate {y[tr_idx].mean():.3f}')
print(f'validation: {len(va_idx)} reviews, positive rate {y[va_idx].mean():.3f}')

## 5. Training

The vectorizer is **re-fitted on the training part only**, so the validation reviews stay completely unseen. A logistic regression is then trained on the TF-IDF vectors. Logistic regression computes `p(positive) = sigmoid(w · x + b)`, with one weight per vocabulary word, trained with cross-entropy loss and L2 regularisation (`C=1`).

In [ ]:
vec_tr = TfidfVectorizer().fit(X_text[tr_idx])
X_tr = vec_tr.transform(X_text[tr_idx])
X_va = vec_tr.transform(X_text[va_idx])

clf = LogisticRegression(max_iter=1000, random_state=SEED)
clf.fit(X_tr, y[tr_idx])
print('features:', X_tr.shape[1], '| train accuracy:', round(accuracy_score(y[tr_idx], clf.predict(X_tr)), 4))

### What the model learned: the most positive and most negative words

In [ ]:
words = vec_tr.get_feature_names_out()
coef = pd.Series(clf.coef_[0], index=words).sort_values()
top = pd.concat([coef.head(15), coef.tail(15)])

fig, ax = plt.subplots(figsize=(9, 7))
ax.barh(top.index, top.values, color=[NEG if v < 0 else POS for v in top.values], height=0.65)
ax.axvline(0, color=MUTED, linewidth=1)
ax.set_title('Top 15 words pushing toward negative (orange) and positive (blue)', loc='left', color=INK)
ax.set_xlabel('Logistic regression weight'); ax.grid(axis='y', visible=False)
plt.tight_layout(); plt.show()

## 6. Validation results

In [ ]:
pred_va = clf.predict(X_va)
p_va = clf.predict_proba(X_va)[:, 1]
acc = accuracy_score(y[va_idx], pred_va)
print(f'Validation accuracy : {acc:.4f}')
print(f'Majority baseline   : {y[va_idx].mean():.4f}')
print(f'Macro-F1            : {f1_score(y[va_idx], pred_va, average="macro"):.4f}')
print(f'ROC-AUC             : {roc_auc_score(y[va_idx], p_va):.4f}\n')
print(classification_report(y[va_idx], pred_va, target_names=['negative', 'positive'], digits=4))

In [ ]:
cm = confusion_matrix(y[va_idx], pred_va)
fpr, tpr, _ = roc_curve(y[va_idx], p_va)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
ax = axes[0]
ax.imshow(cm, cmap='Blues', vmin=0)
ax.grid(False)
for (r, c), v in np.ndenumerate(cm):
    ax.text(c, r, f'{v}\n({v / cm[r].sum():.0%} of row)', ha='center', va='center',
            color='white' if v > cm.max() / 2 else INK)
ax.set_xticks([0, 1], ['Pred. negative', 'Pred. positive']); ax.set_yticks([0, 1], ['Actual negative', 'Actual positive'])
ax.set_title('Confusion matrix (validation)', loc='left', color=INK)

ax = axes[1]
ax.plot(fpr, tpr, color=POS, linewidth=2, label=f'AUC = {roc_auc_score(y[va_idx], p_va):.3f}')
ax.plot([0, 1], [0, 1], color=MUTED, linewidth=1, linestyle='--', label='random guess')
ax.set_title('ROC curve (validation)', loc='left', color=INK)
ax.set_xlabel('False positive rate'); ax.set_ylabel('True positive rate'); ax.legend(frameon=False, loc='lower right')
plt.tight_layout(); plt.show()

**Reading the results:** accuracy is high, but most of it comes from the positive class. The model **misses most negative reviews** (low negative recall in the top-left row) because 85% of the training data is positive. The default 0.5 decision threshold therefore favours "positive". The high ROC-AUC shows the scores do separate the classes well, so class weighting or a lower threshold should improve negative recall.

## 7. 5-fold cross-validation (a more reliable accuracy estimate)

In [ ]:
rows = []
for k, (tr, va) in enumerate(folds):
    v = TfidfVectorizer().fit(X_text[tr])
    m = LogisticRegression(max_iter=1000, random_state=SEED).fit(v.transform(X_text[tr]), y[tr])
    p = m.predict(v.transform(X_text[va]))
    rows.append({'fold': k, 'n_val': len(va), 'accuracy': accuracy_score(y[va], p),
                 'macro_f1': f1_score(y[va], p, average='macro'), 'majority_baseline': y[va].mean()})
cv = pd.DataFrame(rows)
print(f"CV accuracy {cv.accuracy.mean():.4f} ± {cv.accuracy.std():.4f}   |   "
      f"CV macro-F1 {cv.macro_f1.mean():.4f} ± {cv.macro_f1.std():.4f}")

fig, ax = plt.subplots(figsize=(9, 3.4))
ax.plot(cv.fold, cv.accuracy, 'o-', color=POS, linewidth=2, markersize=8, label='TF-IDF + LR accuracy')
ax.plot(cv.fold, cv.majority_baseline, 's--', color=MUTED, linewidth=1.5, markersize=7, label='Always-positive baseline')
for f, a in zip(cv.fold, cv.accuracy):
    ax.annotate(f'{a:.1%}', (f, a), textcoords='offset points', xytext=(0, 8), ha='center', color=INK)
ax.set_xticks(cv.fold, [f'Fold {k}' for k in cv.fold]); ax.set_ylim(0.8, 0.95)
ax.set_ylabel('Accuracy'); ax.set_title('Accuracy per cross-validation fold', loc='left', color=INK)
ax.legend(frameon=False, loc='lower right'); ax.grid(axis='x', visible=False)
plt.tight_layout(); plt.show()
cv.round(4)

## 8. Error analysis: correctly and incorrectly classified validation reviews

In [ ]:
val = pd.DataFrame({'review': X_text[va_idx], 'actual': y[va_idx], 'predicted': pred_va, 'p_positive': p_va.round(3)})
print('Most confident mistakes:')
wrong = val[val.actual != val.predicted].assign(conf=lambda d: (d.p_positive - 0.5).abs()).sort_values('conf', ascending=False)
wrong.drop(columns='conf').head(8)

In [ ]:
print('Most confident correct predictions (one per class):')
right = val[val.actual == val.predicted]
pd.concat([right[right.actual == 0].nsmallest(3, 'p_positive'), right[right.actual == 1].nlargest(3, 'p_positive')])

## 9. Testing: predict `test.json`

The final model is refitted on **all** of `train.json`, using the vectorizer from section 3 (`X_train_full`, 14,114 features, the same as the example). It then predicts the 1,851 test reviews. There are no labels, so no accuracy can be computed. The predictions are written in the `submission.csv` format (`reviews, predictions`).

In [ ]:
from common import RESULTS, ensure_dirs
ensure_dirs()
final_clf = LogisticRegression(max_iter=1000, random_state=SEED).fit(X_train_full, y)
test_pred = final_clf.predict(X_test)
test_p = final_clf.predict_proba(X_test)[:, 1]

submission = pd.DataFrame({'reviews': test_df['reviews'], 'predictions': test_pred})
submission.to_csv(RESULTS / 'submission_tfidf_baseline.csv', index=False)

tc = pd.Series(test_pred).value_counts().sort_index()
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
ax = axes[0]
bars = ax.bar(['Negative (0)', 'Positive (1)'], tc.values, color=[NEG, POS], width=0.5)
for b, v in zip(bars, tc.values):
    ax.text(b.get_x() + b.get_width() / 2, v + 20, f'{v:,} ({v / tc.sum():.1%})', ha='center', color=INK)
ax.set_title('Predicted classes on test.json', loc='left', color=INK); ax.grid(axis='x', visible=False)
ax = axes[1]
ax.hist(test_p, bins=40, color=POS, edgecolor='white', linewidth=0.5)
ax.axvline(0.5, color=NEG, linewidth=1.5, linestyle='--', label='0.5 threshold')
ax.set_title('Predicted p(positive) on test.json', loc='left', color=INK)
ax.set_xlabel('p(positive)'); ax.set_ylabel('Reviews'); ax.legend(frameon=False); ax.grid(axis='x', visible=False)
plt.tight_layout(); plt.show()

print('Saved', RESULTS / 'submission_tfidf_baseline.csv', submission.shape)
print(f'Train positive rate {y.mean():.1%} vs predicted test positive rate {test_pred.mean():.1%}')
submission.assign(p_positive=test_p.round(3)).head(10)